# 🎬 VideoFX — gerador rápido de vídeos (estilo MoneyPrinter)

Roda 100% no navegador: narração TTS neural em português, imagens de fundo, legendas sincronizadas, música e efeito Ken Burns.

**Colab:** `Ambiente de execução → Executar tudo` *(ou clique nos botões ▶ de cada célula)*  
**Kaggle:** *File → Import Notebook → from GitHub* + Settings → **Internet: ON**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jushirambo/videofx/blob/main/videofx_colab.ipynb)


## 1️⃣ Instalação (~1 min)


In [ ]:
%%capture
!git clone https://github.com/Jushirambo/videofx.git /content/videofx
%cd /content/videofx
!pip install -q -r requirements.txt
import os, sys
sys.path.insert(0, '/content/videofx')
print('✅ pronto'))


## 2️⃣ Configure o vídeo
Edite os campos abaixo e rode a célula (▶).
Deixe `ROTULO` vazio para gerar roteiro automático; ou cole suas falas, uma por cena.


In [ ]:
#@title ⚙️ Configuração do vídeo { display-mode: "form" }
topic = "Fatos incríveis sobre o universo" #@param {type:"string"}
title = "FATOS DO UNIVERSO" #@param {type:"string"}
n_scenes = 5 #@param {type:"slider", min:2, max:6, step:1}
aspect = "9:16" #@param ["9:16", "16:9", "1:1"]
music_mood = "calm" #@param ["calm", "energetic"]
ROTULO = "" #@param {type:"string"}

import json, os
cfg = {
    "topic": topic,
    "title": title or None,
    "n_scenes": n_scenes,
    "aspect": aspect,
    "music_mood": music_mood,
    "allow_ai_images": True,
    "output_name": "video_final.mp4",
}
if ROTULO.strip():
    cfg["scenes"] = [l.strip() for l in ROTULO.splitlines() if l.strip()]
    cfg.pop("n_scenes", None)
json.dump(cfg, open("config.json", "w"), ensure_ascii=False, indent=2)
print("config salva ✅", ("(roteiro manual)" if ROTULO.strip() else "(roteiro automático)"))


## 3️⃣ Gere o vídeo (~2–4 min)
 imagem + narracão + música + legendas são geradas nesta célula.


In [ ]:
!python main.py --config config.json


## 4️⃣ Assista e baixe
O vídeo aparece abaixo e um download é disparado no navegador (no Kaggle, use o painel *Output*).


In [ ]:
from IPython.display import Video, display
display(Video("out/video_final.mp4", embed=True, height=560, width=320))
try:
    from google.colab import files
    files.download("out/video_final.mp4")
except ImportError:
    print("Kaggle: baixe em Data ▸ Output ▸ video_final.mp4"))


## 🎛️ (Opcional) Interface interativa no navegador
Rode para abrir um painel Gradio com link público (`*.gradio.live`) durante a sessão.


In [ ]:
%%capture
!pip install -q gradio


In [ ]:
import json, subprocess, gradio as gr, os

def gerar(topico, titulo, aspect, mood, chars, subsize, rotulo):
    cfg = {"topic": topico, "aspect": aspect, "music_mood": mood,
           "subtitle_max_chars": int(chars), "subtitle_size": int(subsize),
           "output_name": "video_final.mp4"}
    if titulo.strip(): cfg["title"] = titulo
    if rotulo.strip(): cfg["scenes"] = [l.strip() for l in rotulo.splitlines() if l.strip()]
    json.dump(cfg, open("config.json", "w"), ensure_ascii=False)
    p = subprocess.run(["python", "main.py", "--config", "config.json"], capture_output=True, text=True)
    if p.returncode != 0:
        raise gr.Error("falhou: " + p.stderr[-300:])
    return "out/video_final.mp4", p.stdout[-500:]

iface = gr.Interface(
    fn=gerar,
    inputs=[gr.Textbox(label="Tema", value="Fatos incríveis sobre o universo"),
            gr.Textbox(label="Título (opcional)"),
            gr.Radio(["9:16", "16:9", "1:1"], value="9:16", label="Formato"),
            gr.Radio(["calm", "energetic"], value="calm", label="Música"),
            gr.Slider(18, 44, value=26, step=1, label="Máx. caracteres da legenda"),
            gr.Slider(40, 96, value=64, step=4, label="Tamanho da legenda"),
            gr.Textbox(lines=4, label="ROTULO (uma fala por linha; vazio = automático)")],
    outputs=[gr.Video(label="Vídeo gerado"), gr.Textbox(label="Log")],
    title="🎬 VideoFX",
)
iface.launch(share=True, debug=True)


## ℹ️ Como funciona
Roteiro (automático ou ROTULO) → TTS neural pt‑BR (Piper, voz baixada no 1º run) → imagem por cena (IA gratuita / fundo procedural) → ffmpeg: Ken Burns + legendas sincronizadas + título + música procedural.

Código: [github.com/Jushirambo/videofx](https://github.com/Jushirambo/videofx)
